# E4 Colab — Qwen3.5-27B bf16 + vLLM (A100 80GB)

로컬 E4(nf4 4bit, transformers)가 val 672에서 **96.88%**를 냈습니다.
이 노트북은 같은 모델을 **bf16 그대로 vLLM**으로 올려 test 6,714문항을 빠르게 풉니다.

| | 로컬 E4 | 이 노트북 |
| --- | --- | --- |
| 정밀도 | nf4 4bit | **bf16** |
| 추론 엔진 | transformers, 1문항씩 | **vLLM, 묶어서** |
| 프롬프트·이미지 규칙·파서 | `common_eval` | 같은 `common_eval` (Drive 사본) |

엔진과 정밀도가 바뀌므로 **먼저 val 672를 돌려(약 5분) 로컬 E4와 맞춰 봅니다.**
정확도가 크게 벗어나거나 일치율이 낮으면 test로 넘어가지 말고 원인을 보세요.

## 준비물 (Drive)

- `MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip` — E1 Colab과 같은 데이터
- `MyDrive/SSAFY/AI_Challenge2/src/common_eval.py` — 로컬 `src/common_eval.py` 사본

val 672 ID와 로컬 E4 예측은 이 노트북 안에 들어 있어 따로 올릴 필요가 없습니다.

## 시간 (A100 80GB 예상)

설치 3~5분 → 모델 다운로드·적재 5~10분 → val 약 5분 → test **40~50분**.
27B는 문항당 약 1,000토큰을 prefill하므로 A100에서도 계산량이 병목입니다.
256문항마다 Drive에 저장하므로 세션이 끊겨도 다시 실행하면 이어집니다.

## 1. 설치

**새 런타임에서 가장 먼저 실행하세요.** vLLM이 torch를 자기 버전으로 바꿉니다.
설치 뒤 import에서 오류가 나면 *런타임 → 세션 다시 시작* 후 2번 셀부터 실행하세요(설치는 유지됩니다).

In [ ]:
!pip -q install -U vllm
!pip -q install -U "huggingface_hub[hf_xet]" pillow pandas

In [ ]:
import gc, json, math, os, re, shutil, sys, time
from pathlib import Path

import pandas as pd
import torch

import vllm
import transformers

print("vllm", vllm.__version__, "/ transformers", transformers.__version__, "/ torch", torch.__version__)
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요 (런타임 - 런타임 유형 변경)."
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} / {GPU_GIB:.1f} GiB")
assert GPU_GIB >= 70, "27B bf16(약 52 GiB)은 A100 80GB가 필요합니다. 40GB면 로컬 4bit 제출을 쓰세요."

## 2. Drive · 데이터 · 공통 코드

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")
ZIP_PATH = DRIVE_BASE / "ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
assert ZIP_PATH.is_file(), f"zip을 찾지 못했습니다: {ZIP_PATH}"


def find_data_root():
    for parent, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d not in {"drive", ".cache", "sample_data", "models"}]
        if "test.csv" in files and "train.csv" in files:
            return Path(parent)
    return None


if find_data_root() is None:
    import subprocess
    subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", "-o/content/", str(ZIP_PATH)],
                   check=True, stdout=subprocess.DEVNULL)
DATA_ROOT = find_data_root()
assert DATA_ROOT is not None, "압축 해제 후에도 train.csv/test.csv를 찾지 못했습니다."

SRC_DIR = Path("/content/src")
SRC_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_SRC = DRIVE_BASE / "src" / "common_eval.py"
assert DRIVE_SRC.is_file(), f"로컬 src/common_eval.py를 {DRIVE_SRC} 로 복사하세요"
shutil.copy2(DRIVE_SRC, SRC_DIR / "common_eval.py")
sys.path.insert(0, str(SRC_DIR))
import common_eval as ce

ce.DATA_DIR = DATA_ROOT
for name in ("SYSTEM", "make_prompt", "parse_choice", "open_image", "IMAGE_MAX_EDGE", "ids_sha256"):
    assert hasattr(ce, name), f"Drive의 common_eval.py가 오래됐습니다: {name} 없음"
print("데이터:", DATA_ROOT, "/ 이미지 긴 변 제한:", ce.IMAGE_MAX_EDGE, "px")

## 3. 모델 적재 (vLLM)

`MODEL_ID`를 넘기면 vLLM이 Hugging Face에서 바로 받습니다(52 GiB, 보통 수 분).
이미지는 문항당 1장이고 프롬프트는 1,000토큰 안팎이라 `max_model_len`은 4,096이면 충분합니다.

In [ ]:
!pip -q uninstall -y torchaudio

In [ ]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3.5-27B"
MAX_NEW_TOKENS = 24          # 로컬 E1~E4와 동일
LOGPROBS_K = 20              # 첫 토큰 상위 k개 로그확률. a~d 확률을 뽑는 데 씁니다

t0 = time.time()
llm = LLM(model=MODEL_ID, dtype="bfloat16", max_model_len=4096, gpu_memory_utilization=0.90,
          limit_mm_per_prompt={"image": 1}, seed=42, download_dir="/content/models")
proc = AutoProcessor.from_pretrained(MODEL_ID)
print(f"적재 {time.time() - t0:.0f}초")

SAMPLING = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, logprobs=LOGPROBS_K, seed=42)


def letter_token_ids():
    tok = proc.tokenizer
    table = {}
    for ch in ce.CHOICES:
        cand = {tok.encode(f, add_special_tokens=False)[0]
                for f in (ch, " " + ch, ch.upper(), " " + ch.upper())
                if tok.encode(f, add_special_tokens=False)}
        table[ch] = sorted(cand)
    return table


LETTERS = letter_token_ids()


def make_request(row):
    """로컬과 같은 메시지를 HF 채팅 템플릿으로 렌더링하고, 이미지는 같은 open_image로 줄여 넘깁니다."""
    image = ce.open_image(row)
    messages = [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": ce.make_prompt(row)}]}]
    prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                      enable_thinking=False)
    return {"prompt": prompt, "multi_modal_data": {"image": image}}, image.size

print("a~d 토큰 id:", LETTERS)

## 4. 추론 함수

묶음 단위로 vLLM에 넘기고, 묶음이 끝날 때마다 `predictions.jsonl`에 쓰고 Drive에 복사합니다.
다시 실행하면 기록된 문항을 건너뜁니다. 로컬과 같이 `parse_choice`로 답을 읽고,
첫 토큰의 a~d 로그확률도 함께 남깁니다(상위 20개에 없으면 `-1e9`).

In [ ]:
CHUNK = 256


def letter_logprobs(first_step):
    out = {}
    for ch, ids in LETTERS.items():
        vals = [first_step[i].logprob for i in ids if i in first_step]
        out[ch] = max(vals) if vals else -1e9   # 상위 k개 밖. JSON에 -inf를 쓰지 않으려고 유한값
    return out


def run(rows, out_dir, drive_dir, desc):
    out_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)
    pred_path = out_dir / "predictions.jsonl"
    if not pred_path.is_file() and (drive_dir / "predictions.jsonl").is_file():
        shutil.copy2(drive_dir / "predictions.jsonl", pred_path)
        print("Drive에서 이전 기록을 복구했습니다")
    done = set()
    if pred_path.is_file():
        done = {json.loads(l)["id"] for l in pred_path.open(encoding="utf-8") if l.strip()}
    todo = [r for r in rows if r["id"] not in done]
    print(f"[{desc}] 대상 {len(rows)} / 기록됨 {len(done)} / 남은 {len(todo)}")

    t_all = time.time()
    for start in range(0, len(todo), CHUNK):
        batch = todo[start:start + CHUNK]
        t0 = time.time()
        reqs, sizes = zip(*(make_request(r) for r in batch))
        if start == 0:
            assert "<|image_pad|>" in reqs[0]["prompt"], "프롬프트에 이미지 토큰이 없습니다"
        outs = llm.generate(list(reqs), SAMPLING, use_tqdm=False)
        elapsed = time.time() - t0
        with pred_path.open("a", encoding="utf-8") as log:
            for row, size, o in zip(batch, sizes, outs):
                c = o.outputs[0]
                raw = c.text.strip()
                log.write(json.dumps(dict(
                    id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),
                    letter_logprobs=letter_logprobs(c.logprobs[0]) if c.logprobs else {},
                    answer_truncated=c.finish_reason == "length", answer_tokens=len(c.token_ids),
                    seconds=elapsed / len(batch), image_width=size[0], image_height=size[1]),
                    ensure_ascii=False) + "\n")
        for f in out_dir.glob("*"):
            if f.is_file():
                shutil.copy2(f, drive_dir / f.name)
        n_done = start + len(batch)
        rate = (time.time() - t_all) / n_done
        print(f"  {n_done}/{len(todo)}  {elapsed / len(batch):.3f}초/문항  "
              f"남은 약 {rate * (len(todo) - n_done) / 60:.0f}분")
    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")
    return res

## 5. val 672 — 로컬 E4와 맞춰 보기

로컬 E4와 **같은 672문항**입니다(ID SHA-256 `0aad93d6210e1b23…` 대조).
로컬 E4의 문항별 예측도 노트북에 들어 있어 일치율을 바로 봅니다.

판단 기준: 정확도가 로컬(96.88%)에서 ±1%p 안이고 일치율이 95% 이상이면 test로 넘어갑니다.
로컬 E4와 E1(9B)의 불일치가 4.5%였으므로, 같은 모델의 정밀도 차이는 그보다 작아야 정상입니다.

In [ ]:
VAL_NUMS = [37, 57, 65, 84, 91, 108, 113, 119, 124, 130, 132, 134, 145, 169, 174, 177, 193, 213, 221, 224, 225, 247, 251, 267, 273, 281, 300, 307, 309, 316, 317, 322, 356, 360, 363, 370, 388, 390, 401, 428, 432, 442, 481, 500, 508, 517, 528, 537, 551, 558, 567, 584, 594, 601, 603, 613, 641, 649, 650, 670, 673, 675, 698, 703, 756, 775, 778, 785, 787, 792, 797, 803, 807, 827, 833, 842, 846, 858, 863, 870, 871, 873, 893, 909, 912, 913, 937, 944, 968, 983, 985, 1000, 1005, 1006, 1012, 1034, 1039, 1042, 1045, 1049, 1061, 1072, 1079, 1080, 1092, 1094, 1097, 1106, 1117, 1118, 1120, 1127, 1139, 1145, 1149, 1152, 1155, 1156, 1166, 1171, 1172, 1193, 1199, 1200, 1201, 1213, 1272, 1275, 1297, 1298, 1324, 1325, 1328, 1329, 1333, 1341, 1342, 1348, 1361, 1372, 1392, 1402, 1420, 1433, 1437, 1458, 1461, 1471, 1483, 1512, 1525, 1528, 1530, 1533, 1537, 1543, 1560, 1578, 1593, 1614, 1641, 1650, 1659, 1663, 1705, 1725, 1730, 1734, 1744, 1753, 1755, 1758, 1762, 1770, 1803, 1825, 1827, 1828, 1837, 1861, 1867, 1888, 1891, 1892, 1893, 1894, 1903, 1919, 1930, 1931, 1954, 1960, 1977, 1981, 1982, 1992, 1997, 2014, 2018, 2031, 2046, 2049, 2060, 2079, 2085, 2087, 2105, 2120, 2133, 2144, 2149, 2172, 2173, 2177, 2189, 2198, 2206, 2216, 2222, 2241, 2263, 2288, 2308, 2313, 2315, 2324, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2415, 2436, 2449, 2458, 2467, 2470, 2481, 2487, 2498, 2514, 2517, 2527, 2538, 2546, 2548, 2561, 2565, 2583, 2586, 2590, 2628, 2662, 2668, 2669, 2697, 2704, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2771, 2777, 2784, 2797, 2812, 2816, 2817, 2830, 2836, 2852, 2864, 2865, 2907, 2925, 2928, 2931, 2937, 2967, 2972, 2993, 3003, 3011, 3013, 3045, 3054, 3073, 3076, 3083, 3086, 3100, 3113, 3115, 3121, 3128, 3154, 3194, 3199, 3202, 3228, 3230, 3239, 3244, 3250, 3266, 3280, 3289, 3298, 3323, 3328, 3333, 3349, 3359, 3361, 3364, 3374, 3383, 3408, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3470, 3489, 3541, 3549, 3580, 3582, 3583, 3594, 3599, 3616, 3622, 3623, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3688, 3696, 3698, 3699, 3718, 3757, 3792, 3802, 3809, 3821, 3832, 3875, 3888, 3890, 3899, 3912, 3916, 3920, 3930, 3943, 3947, 3973, 3977, 3981, 3985, 3996, 4018, 4039, 4044, 4068, 4074, 4082, 4106, 4117, 4125, 4130, 4132, 4149, 4167, 4186, 4191, 4193, 4195, 4204, 4208, 4213, 4229, 4254, 4257, 4264, 4267, 4291, 4293, 4297, 4305, 4306, 4310, 4315, 4332, 4341, 4354, 4367, 4372, 4380, 4392, 4398, 4401, 4402, 4404, 4408, 4409, 4419, 4430, 4439, 4445, 4450, 4454, 4461, 4468, 4471, 4478, 4491, 4497, 4518, 4528, 4531, 4540, 4553, 4569, 4584, 4596, 4600, 4602, 4603, 4605, 4614, 4630, 4639, 4649, 4657, 4659, 4668, 4670, 4701, 4708, 4719, 4720, 4744, 4748, 4754, 4760, 4764, 4771, 4786, 4789, 4790, 4801, 4802, 4808, 4812, 4814, 4822, 4824, 4831, 4836, 4843, 4847, 4859, 4866, 4868, 4874, 4878, 4889, 4914, 4917, 4918, 4925, 4944, 4954, 4968, 4971, 5009, 5013, 5045, 5058, 5077, 5080, 5109, 5126, 5145, 5148, 5155, 5158, 5167, 5169, 5182, 5187, 5188, 5209, 5220, 5245, 5254, 5261, 5266, 5268, 5269, 5285, 5289, 5290, 5308, 5331, 5340, 5361, 5371, 5374, 5384, 5420, 5422, 5452, 5459, 5461, 5478, 5486, 5488, 5495, 5499, 5506, 5509, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5545, 5547, 5555, 5556, 5560, 5564, 5610, 5618, 5622, 5629, 5653, 5683, 5684, 5723, 5729, 5730, 5747, 5789, 5790, 5811, 5818, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5867, 5880, 5885, 5888, 5895, 5904, 5906, 5928, 5936, 5950, 5962, 5972, 5982, 5992, 6017, 6025, 6031, 6037, 6040, 6050, 6069, 6100, 6101, 6113, 6131, 6143, 6149, 6154, 6157, 6160, 6162, 6169, 6171, 6179, 6189, 6196, 6212, 6216, 6217, 6241, 6259, 6260, 6267, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6359, 6371, 6375, 6378, 6390, 6404, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6464, 6476, 6522, 6528, 6547, 6548, 6549, 6551, 6556, 6576, 6584, 6590, 6603, 6611, 6618, 6637, 6645, 6647, 6663, 6671, 6673, 6675, 6684, 6693, 6702, 6703]
VAL_IDS = [f"train_{n:04d}.jpg" for n in VAL_NUMS]
assert ce.ids_sha256(VAL_IDS) == "0aad93d6210e1b23f94186edebead4cd754bf0f93609d59f06f67c5489791cfe", "val ID 해시가 다릅니다"
E4_LOCAL = "abbdbdbccbcbddcbddabddcbddacaccdbbadcadbdccaabbadcbabdabadbacbabcabdbdbcadbbccccccbddcaababbcaabbbcbacdbacbbbddcadbbbaacddbadabdacdcbadaabdddbddccbcbabdbadacdbddcccdaaabacbbabbcbcdabccaacdbcdddbdacacaddcdcdcccdddcbdaaddcbcacbabcbdbbbdccaaccabcbcabaabbccadbcaccdbcddbaacbbaddabbcbaacccdcbadbbaaddcbababcdadbaacccabbbccaddccddabbdbdcbdddaaaacbaabcbbaacbbdadddbccaaaaaacaacccbdcdcacdddddbaaaccdcdcbbdddbddbbdcbcbcacccdcccdcdacacbbbaddbcaddbbcadabcabbcdacaccdbdaaacdbddaacddadaddbcacbbccccbadbdbbbdbaadbababbaabaadccbdabdacbacaacdacccbadacbadbccbccdcbadbddaacccccdbbcaadcbaadcabcccdcbabdddcbdcccddbaddabdcaaaaccaabcaddcaaddacccdcddccabacddaaccdddbdabdabbbbbcbccaadbccabaddcacb"   # 로컬 E4(nf4) 예측, VAL_IDS 순서
E4_LOCAL_ACC = 0.96875

train_df = pd.read_csv(DATA_ROOT / "train.csv", keep_default_na=False).set_index("id")
val_df = train_df.loc[VAL_IDS].reset_index()
val_rows = val_df.drop(columns=["answer"]).to_dict("records")

VAL_OUT = Path("/content/out/exp_E4_vllm_val")
VAL_DRIVE = DRIVE_BASE / "output" / "exp_E4_vllm_val"
val_res = run(val_rows, VAL_OUT, VAL_DRIVE, "val 672")

v = val_df[["id", "answer"]].merge(val_res[["id", "pred"]], on="id", validate="one_to_one")
v["local"] = list(E4_LOCAL)
acc = (v.pred == v.answer).mean()
agree = (v.pred == v.local).mean()
b = int(((v.local == v.answer) & (v.pred != v.answer)).sum())
c = int(((v.local != v.answer) & (v.pred == v.answer)).sum())
print(f"\nvLLM bf16 {acc:.4f} ({int((v.pred == v.answer).sum())}/672) / 로컬 nf4 {E4_LOCAL_ACC:.4f}")
print(f"문항별 일치율 {agree:.1%} / 로컬만 정답 {b} / vLLM만 정답 {c} / INVALID {int((v.pred == 'INVALID').sum())}")
OK_TO_SUBMIT = abs(acc - E4_LOCAL_ACC) <= 0.01 and agree >= 0.95
print("-> test로 진행하세요" if OK_TO_SUBMIT else
      "-> 로컬과 차이가 큽니다. test로 넘어가기 전에 원인을 확인하세요 (INVALID·프롬프트·이미지 토큰)")

## 6. test 6,714 — 제출 파일

위 판단이 `test로 진행하세요`일 때만 실행하세요. 파서가 실패한 문항은 기록된 a~d 로그확률 중
가장 높은 글자로 채웁니다(특정 글자로 메우면 분포가 쏠립니다).

In [ ]:
assert OK_TO_SUBMIT, "val 확인을 통과하지 못했습니다. 그래도 진행하려면 OK_TO_SUBMIT = True로 두세요"

test_df = pd.read_csv(DATA_ROOT / "test.csv", keep_default_na=False)
sample_df = pd.read_csv(DATA_ROOT / "sample_submission.csv", keep_default_na=False)
assert list(test_df.id) == list(sample_df.id), "sample_submission과 id 순서가 다릅니다"
assert test_df.id.is_unique

TEST_OUT = Path("/content/out/exp_E4_vllm_test")
TEST_DRIVE = DRIVE_BASE / "output" / "exp_E4_vllm_test"
test_res = run(test_df.to_dict("records"), TEST_OUT, TEST_DRIVE, "test 6,714")

In [ ]:
res = test_res.copy()
fallback = res.pred == "INVALID"
res.loc[fallback, "pred"] = res.loc[fallback, "letter_logprobs"].map(
    lambda lp: max(lp, key=lp.get) if lp else "INVALID")
sub = test_df[["id"]].merge(res[["id", "pred"]], on="id", how="left", validate="one_to_one")
assert sub.pred.isin(ce.CHOICES).all(), f"a~d 이외의 값: {sorted(set(sub.pred) - set(ce.CHOICES))}"
assert list(sub.id) == list(sample_df.id)

share = sub.pred.value_counts(normalize=True).reindex(ce.CHOICES).fillna(0)
print(f"폴백 {int(fallback.sum())}건 / 예측 분포"); print(share.round(3).to_string())
if share.max() > 0.35:
    print(f"경고: '{share.idxmax()}'가 {share.max():.1%}로 쏠렸습니다")

stem = "submission_qwen35-27b_zeroshot-bf16-vllm"
used = [int(m.group(1)) for p in TEST_DRIVE.glob(stem + "_v*.csv")
        if (m := re.fullmatch(re.escape(stem) + r"_v(\d+)", p.stem))]
name = f"{stem}_v{max(used, default=0) + 1}.csv"
sub.rename(columns={"pred": "answer"}).to_csv(TEST_OUT / name, index=False, encoding="utf-8")
shutil.copy2(TEST_OUT / name, TEST_DRIVE / name)
print("제출 파일:", TEST_DRIVE / name)

## 끝나면

`MyDrive/SSAFY/AI_Challenge2/output/exp_E4_vllm_val`, `exp_E4_vllm_test`를 로컬 `output/`으로 받아 두세요.
val 결과는 bf16과 nf4의 차이를, test 예측은 이후 로컬 4bit 제출이나 E1 제출과의 비교에 씁니다.